# Functional Regression: Input Time Series → Output Time Series

**Pipeline:**
1. Feature-engineer the input time series (raw values + first differences + cumulative sum + pairwise param ratios)
2. Sizes adapt automatically to any N_SERIES or N_PARAMS
3. Compress the output with PCA (retain 98% variance)
4. Train `GradientBoostingRegressor` wrapped in `MultiOutputRegressor` on PCA scores
5. Reconstruct full output via inverse PCA
6. Optional: smooth output curves with UnivariateSpline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from scipy.interpolate import UnivariateSpline

## Data
Replace the placeholder arrays with real data. Keep the shape conventions:
- `X_series`: `(N, 11)` — input time series, one row per sample
- `X_params`: `(N, 3)` — scalar conditioning parameters
- `Y_raw`: `(N, 1500)` — target output time series

In [ ]:
N = 8  # number of samples  <-- update when you have real data

# Time axes (adjust step / span to match your data)
INPUT_TIME  = np.linspace(0, 1, 11)    # 11-pt input time axis
OUTPUT_TIME = np.linspace(0, 1, 1500)  # 1500-pt output time axis


In [ ]:
X_series = np.load('X_series.npy')
X_params = np.load('X_params.npy')
Y_raw=np.load('Y_raw.npy')

print(X_series.shape)
print(X_params.shape)
print(Y_raw.shape)

## Input Feature Engineering
Augment the input time series with temporal features and generate new variables from scalar params.
Sizes are inferred automatically from data shape — add more columns to `X_params` and everything adapts.

| Group | Size | Captures |
|---|---|---|
| Raw values | N_SERIES | Amplitude at each time step |
| First differences (Δ) | N_SERIES − 1 | Rate of change |
| Cumulative sum | N_SERIES | Integral / trend |
| Scalar params | N_PARAMS | Conditioning |
| Pairwise param ratios | N_PARAMS·(N_PARAMS−1)/2 | Relative relationships between params |
| **Total** | **3·N_SERIES − 1 + N_PARAMS + ratios** | |

In [ ]:
def make_features(X_series, X_params):
    """Build feature matrix from raw series and scalar params.
    Works for any N_SERIES or N_PARAMS — sizes inferred from data shape.
    """
    eps = 1e-8
    series_raw  = X_series                          # (N, N_SERIES)
    series_diff = np.diff(X_series, axis=1)         # (N, N_SERIES-1)
    series_cum  = np.cumsum(X_series, axis=1)       # (N, N_SERIES)

    n_p = X_params.shape[1]
    ratio_list = [
        X_params[:, i:i+1] / (X_params[:, j:j+1] + eps)
        for i, j in combinations(range(n_p), 2)
    ]
    param_ratios = np.hstack(ratio_list) if ratio_list else np.empty((X_series.shape[0], 0))

    return np.hstack([series_raw, series_diff, series_cum, X_params, param_ratios])


X_feat = make_features(X_series, X_params)

# --- Index slices (for use in later cells) ---
n_series = X_series.shape[1]
n_params = X_params.shape[1]
n_ratios = n_params * (n_params - 1) // 2
n_total  = X_feat.shape[1]

idx_raw    = slice(0, n_series)
idx_diff   = slice(n_series, 2 * n_series - 1)
idx_cum    = slice(2 * n_series - 1, 3 * n_series - 1)
idx_params = slice(3 * n_series - 1, 3 * n_series - 1 + n_params)
idx_ratios = slice(3 * n_series - 1 + n_params, None)

feature_names = (
    [f"raw_{i}"   for i in range(n_series)] +
    [f"diff_{i}"  for i in range(n_series - 1)] +
    [f"cum_{i}"   for i in range(n_series)] +
    [f"param_{i}" for i in range(n_params)] +
    [f"p{i}/p{j}" for i, j in combinations(range(n_params), 2)]
)

print(f"N_SERIES={n_series}, N_PARAMS={n_params}, N_RATIOS={n_ratios}")
print(f"X_feat shape: {X_feat.shape}  ({n_total} features total)")

In [ ]:
## Correlation Analysis

Y_max = Y_raw.max(axis=1)  # peak output per sample

# Correlation of each input feature with Y_max
corr_with_Ymax = np.array([
    np.corrcoef(X_feat[:, f], Y_max)[0, 1]
    for f in range(len(feature_names))
])

# Sort by absolute correlation
sort_idx = np.argsort(np.abs(corr_with_Ymax))[::-1]
sorted_names = [feature_names[i] for i in sort_idx]
sorted_corr  = corr_with_Ymax[sort_idx]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if c >= 0 else 'tomato' for c in sorted_corr]
ax.bar(range(len(sorted_names)), sorted_corr, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(sorted_names)))
ax.set_xticklabels(sorted_names, rotation=90, fontsize=8)
ax.set_ylabel('Pearson r with Y_max')
ax.set_title('Input Feature Correlation with Y_max (sorted by |r|)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Train / Test Split
Split **before** fitting any scaler or PCA to avoid data leakage.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_feat, Y_raw, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

## Normalization Scaffold
`scaler_X` is active now and is **required** for deep learning later.  
`scaler_Y` is commented out — **activate** when switching to a neural network.

In [ ]:
# Input normalization (used by both ML and DL)
scaler_X = StandardScaler()
X_scaled_train = scaler_X.fit_transform(X_train)
X_scaled_test  = scaler_X.transform(X_test)

# Output normalization — activate for deep learning
# scaler_Y = StandardScaler()
# Y_scaled_train = scaler_Y.fit_transform(Y_train)
# Y_scaled_test  = scaler_Y.transform(Y_test)

## PCA on Output
Compresses the 1500-dimensional output to `ky` principal components (95% variance retained).  
This reduces the effective regression target from 1500 scalars to ~30–80, greatly improving accuracy on small N.

In [ ]:
pca_y = PCA(n_components=0.98)
Z_y_train = pca_y.fit_transform(Y_train)   # (N_train, ky)
Z_y_test  = pca_y.transform(Y_test)        # (N_test,  ky)

ky = pca_y.n_components_
print(f"Output PCA: {ky} components retain 95% variance (from 1500)")

# Scree plot
cumvar = np.cumsum(pca_y.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, ky + 1), cumvar * 100, marker='o', markersize=3)
ax.axhline(95, color='red', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('Output PCA — Scree plot')
ax.legend()
plt.tight_layout()
plt.show()

## ML Model: GradientBoosting + MultiOutputRegressor
Maps the feature vector (raw + diff + cumsum + params + ratios) to the `ky` PCA scores of the output.
Output is then reconstructed via `pca_y.inverse_transform`.

In [ ]:
base_estimator = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    min_samples_leaf=2,
    random_state=42
)

model = MultiOutputRegressor(base_estimator, n_jobs=-1)
model.fit(X_scaled_train, Z_y_train)

Z_pred = model.predict(X_scaled_test)              # (N_test, ky)
Y_pred = pca_y.inverse_transform(Z_pred)           # (N_test, 1500)

print(f"Y_pred shape: {Y_pred.shape}")

## Optional: Smooth Predicted Output Curves
Applies a cubic smoothing spline to each predicted curve.  
Tune `S_PARAM`: smaller → tighter fit, larger → smoother curve.

In [ ]:
S_PARAM = 0.01  # smoothing parameter — tune to your data

Y_pred_smooth = np.empty_like(Y_pred)
for i in range(len(Y_pred)):
    spl = UnivariateSpline(OUTPUT_TIME, Y_pred[i], k=3, s=S_PARAM)
    Y_pred_smooth[i] = spl(OUTPUT_TIME)

## Deep Learning — Placeholder (commented out)
When ready to switch to a neural network:
1. Activate `scaler_Y` in the normalization cell above
2. Uncomment and implement the network below
3. Keep `scaler_X` and `pca_y` — they remain useful for DL too

In [ ]:
# TODO: replace ML model with a PyTorch / Keras network
#
# Option A — predict PCA scores (keeps inverse_transform step):
#   Input:  X_scaled_train  (N, 35)
#   Target: Z_y_train       (N, ky)   ← pca_y.inverse_transform to get back 1500 pts
#
# Option B — predict full output directly (activate scaler_Y above):
#   Input:  X_scaled_train  (N, 35)
#   Target: Y_scaled_train  (N, 1500) ← scaler_Y.inverse_transform to get real units
#
# Option C — use 1D-CNN / LSTM encoder on raw input series:
#   Input:  X_series (N, 11) fed as sequences + X_params (N, 3) as conditioning
#   Target: Z_y_train (N, ky)  or  Y_scaled_train (N, 1500)
#
# import torch
# import torch.nn as nn
# ...
pass

In [ ]:
## Evaluation

mse_ml  = mean_squared_error(Y_test, Y_pred)
r2_ml   = r2_score(Y_test, Y_pred)
mse_spl = mean_squared_error(Y_test, Y_pred_smooth)
r2_spl  = r2_score(Y_test, Y_pred_smooth)

print(f"ML prediction   — MSE: {mse_ml:.6f}  R²: {r2_ml:.4f}")
print(f"Spline smoothed — MSE: {mse_spl:.6f}  R²: {r2_spl:.4f}")

# R² per test sample (computed across 1500 time points — statistically meaningful)
# Note: R² per time step requires many test samples; use per-sample R² for small N
r2_per_sample = np.array([
    r2_score(Y_test[i], Y_pred[i])
    for i in range(len(Y_test))
])
print(f"\nPer-sample R²: {np.round(r2_per_sample, 4)}")
print(f"Mean: {r2_per_sample.mean():.4f}  |  Min: {r2_per_sample.min():.4f}")

SAMPLE_IDX = 0

fig, axes = plt.subplots(4, 1, figsize=(12, 22))

# Input time series
ax = axes[0]
ax.plot(INPUT_TIME, X_test[SAMPLE_IDX, idx_raw], marker='o', label='Input series')
ax.set_title(f'Sample {SAMPLE_IDX} — Input time series')
ax.set_xlabel('Input time')
ax.legend()
ax.grid(True, alpha=0.3)

# Scalar params
ax = axes[1]
param_vals = X_test[SAMPLE_IDX, idx_params]
ax.bar(range(n_params), param_vals)
ax.set_xticks(range(n_params))
ax.set_xticklabels([f"param_{i}" for i in range(n_params)])
ax.set_title(f'Sample {SAMPLE_IDX} — Scalar params')
ax.grid(True, alpha=0.3)

# Output: true vs predicted vs smoothed
ax = axes[2]
ax.plot(OUTPUT_TIME, Y_test[SAMPLE_IDX],        label='True output',   linewidth=2)
ax.plot(OUTPUT_TIME, Y_pred[SAMPLE_IDX],        label='ML prediction', linewidth=2, linestyle='--')
ax.plot(OUTPUT_TIME, Y_pred_smooth[SAMPLE_IDX], label='Smoothed',      linewidth=2, linestyle=':')
ax.set_title(f'Sample {SAMPLE_IDX} — Output prediction')
ax.set_xlabel('Time')
ax.set_ylabel('Output')
ax.legend()
ax.grid(True, alpha=0.3)

# R² per test sample
ax = axes[3]
ax.bar(range(len(Y_test)), r2_per_sample, color=['steelblue' if r > 0 else 'tomato' for r in r2_per_sample])
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(Y_test)))
ax.set_xticklabels([f"test {i}" for i in range(len(Y_test))])
ax.set_title('R² per test sample (across 1500 output time points)')
ax.set_ylabel('R²')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# New query

In [ ]:
# Temperature sensitivity: raise hot zone by 10°C
X_series_original = np.array([156.5, 173.7, 183.1, 180.2, 182.1, 196.4, 249.3, 258.6, 259.3, 130.8, 116.6])
X_series_changed  = np.array([156.5, 173.7, 183.1, 180.2, 182.1, 196.4, 259.3, 268.6, 269.3, 130.8, 116.6])
X_params_q        = np.array([9940, 13090, 5.57])

X_series_original = X_series_original.reshape(1, -1)
X_series_changed  = X_series_changed.reshape(1, -1)
X_params_q        = X_params_q.reshape(1, -1)

X_feat_original = make_features(X_series_original, X_params_q)
X_feat_changed  = make_features(X_series_changed,  X_params_q)

X_feat_original_scaled = scaler_X.transform(X_feat_original)
X_feat_changed_scaled  = scaler_X.transform(X_feat_changed)

Y_pred_original = pca_y.inverse_transform(model.predict(X_feat_original_scaled))
Y_pred_changed  = pca_y.inverse_transform(model.predict(X_feat_changed_scaled))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
ax = axes[0]
ax.plot(INPUT_TIME, X_series_original[0], label='Original')
ax.plot(INPUT_TIME, X_series_changed[0],  label='+10°C hot zone')
ax.set_ylim(0, 300)
ax.set_title('Rail temperature')
ax.set_ylabel('Temperature')
ax.legend()

ax = axes[1]
ax.plot(OUTPUT_TIME, Y_pred_original[0], label='Original')
ax.plot(OUTPUT_TIME, Y_pred_changed[0],  label='+10°C hot zone')
ax.set_title('Predicted PCB Temperature')
ax.set_ylabel('Temperature')
ax.set_xlabel('Time')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Change PCB: compare two different param configurations
X_series_q       = np.array([156.5, 173.7, 183.1, 180.2, 182.1, 196.4, 249.3, 258.6, 259.3, 130.8, 116.6])
X_params_orig_q  = np.array([9200,  22300, 9.2])
X_params_new_q   = np.array([4693,  10052, 4.18])

X_series_q      = X_series_q.reshape(1, -1)
X_params_orig_q = X_params_orig_q.reshape(1, -1)
X_params_new_q  = X_params_new_q.reshape(1, -1)

X_feat_original = make_features(X_series_q, X_params_orig_q)
X_feat_changed  = make_features(X_series_q, X_params_new_q)

X_feat_original_scaled = scaler_X.transform(X_feat_original)
X_feat_changed_scaled  = scaler_X.transform(X_feat_changed)

Y_pred_original = pca_y.inverse_transform(model.predict(X_feat_original_scaled))
Y_pred_changed  = pca_y.inverse_transform(model.predict(X_feat_changed_scaled))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
ax = axes[0]
ax.bar(np.arange(n_params) - 0.2, X_params_orig_q[0], width=0.4, label='Original PCB')
ax.bar(np.arange(n_params) + 0.2, X_params_new_q[0],  width=0.4, label='New PCB')
ax.set_xticks(range(n_params))
ax.set_xticklabels([f"param_{i}" for i in range(n_params)])
ax.set_title('PCB Parameters')
ax.legend()

ax = axes[1]
ax.plot(OUTPUT_TIME, Y_pred_original[0], label='Original PCB')
ax.plot(OUTPUT_TIME, Y_pred_changed[0],  label='New PCB')
ax.set_title('Predicted PCB Temperature')
ax.set_ylabel('Temperature')
ax.set_xlabel('Time')
ax.legend()

plt.tight_layout()
plt.show()